# Tuning & Experiment Tracking

**Week 8 -- CS 203 -- Software Tools and Techniques for AI**

Prof. Nipun Batra, IIT Gandhinagar

---

**Outline:**
1. Grid Search vs Random Search
2. Bayesian Optimization (1D visualization + Optuna + bayesian-optimization)
3. Nested Cross-Validation (the optimism gap)
4. AutoML with FLAML
5. PyTorch Reproducibility
6. Experiment Tracking with Trackio

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install numpy matplotlib scikit-learn optuna bayesian-optimization flaml torch trackio

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import (
    train_test_split, cross_val_score, GridSearchCV,
    RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import accuracy_score
from scipy.stats import randint, uniform
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

In [ ]:
# ---- Synthetic dataset (used throughout Parts 1-4) ----
X, y = make_classification(
    n_samples=1000, n_features=10, n_informative=5,
    n_redundant=2, n_classes=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Dataset : {X.shape[0]} samples, {X.shape[1]} features")
print(f"Train   : {X_train.shape[0]}  |  Test: {X_test.shape[0]}")
print(f"Class balance: {y.mean():.1%} positive")

---
## Part 1: Grid Search vs Random Search

In [ ]:
# --- Grid Search ---
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 2, 5],
}

grid_cv = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    return_train_score=True,
)
grid_cv.fit(X_train, y_train)

BUDGET = len(grid_cv.cv_results_['params'])  # total grid combos
print(f"Grid Search: {BUDGET} combinations evaluated")
print(f"Best CV score : {grid_cv.best_score_:.4f}")
print(f"Best params   : {grid_cv.best_params_}")

In [ ]:
# --- Random Search (same budget) ---
param_distributions = {
    'n_estimators': randint(50, 500),
    'max_depth': randint(3, 30),
    'min_samples_leaf': randint(1, 20),
    'max_features': uniform(0.1, 0.9),
}

random_cv = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions,
    n_iter=BUDGET,
    cv=5,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    return_train_score=True,
)
random_cv.fit(X_train, y_train)

print(f"Random Search: {BUDGET} combinations evaluated")
print(f"Best CV score : {random_cv.best_score_:.4f}")
print(f"Best params   : {random_cv.best_params_}")
print()
diff = random_cv.best_score_ - grid_cv.best_score_
winner = "Random" if diff > 0 else "Grid"
print(f"{winner} Search wins by {abs(diff):.4f}")

In [ ]:
# --- Plot: parameter space explored by each method ---
grid_params = grid_cv.cv_results_['params']
rand_params = random_cv.cv_results_['params']

def extract(params_list, key, default=None):
    vals = []
    for p in params_list:
        v = p.get(key, default)
        vals.append(v if v is not None else 30)
    return np.array(vals, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
g_ne = extract(grid_params, 'n_estimators')
g_md = extract(grid_params, 'max_depth')
g_scores = grid_cv.cv_results_['mean_test_score']
sc = ax.scatter(g_ne, g_md, c=g_scores, cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth (None -> 30)')
ax.set_title(f'Grid Search ({BUDGET} evals)')
plt.colorbar(sc, ax=ax, label='CV Accuracy')

ax = axes[1]
r_ne = extract(rand_params, 'n_estimators')
r_md = extract(rand_params, 'max_depth')
r_scores = random_cv.cv_results_['mean_test_score']
sc = ax.scatter(r_ne, r_md, c=r_scores, cmap='viridis', s=60, edgecolors='k', linewidths=0.5)
ax.set_xlabel('n_estimators')
ax.set_ylabel('max_depth')
ax.set_title(f'Random Search ({BUDGET} evals)')
plt.colorbar(sc, ax=ax, label='CV Accuracy')

fig.suptitle('Parameter Space Explored: Grid vs Random', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Takeaway:** Grid search only evaluates points on a fixed lattice. Random search covers the space more uniformly, which matters when some hyperparameters are more important than others (Bergstra & Bengio, 2012).

---
## Part 2: Bayesian Optimization

### 2a. 1D Visualization: How Bayesian Optimization Works

Let's visualize GP-based Bayesian optimization on a simple 1D function so you can see the surrogate model and acquisition function in action.

In [ ]:
from bayes_opt import BayesianOptimization, UtilityFunction

# A 1D function we want to maximize (optimizer doesn't see the formula)
def target_function(x):
    return np.sin(x) * x + np.cos(2 * x)

# Plot the true function
x_range = np.linspace(0, 6, 300)
y_true = [target_function(x) for x in x_range]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x_range, y_true, 'b-', linewidth=2, label='True function (unknown to optimizer)')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_title('The function we want to maximize')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Run Bayesian Optimization step by step
optimizer = BayesianOptimization(
    f=target_function,
    pbounds={'x': (0, 6)},
    random_state=42,
    verbose=0
)

# Start with 3 random points
optimizer.maximize(init_points=3, n_iter=0)

print("Initial random points:")
for i, res in enumerate(optimizer.res):
    print(f"  x={res['params']['x']:.3f}, f(x)={res['target']:.3f}")

# Now do 7 more iterations with Bayesian optimization
optimizer.maximize(init_points=0, n_iter=7)

print(f"\nBest result: x={optimizer.max['params']['x']:.3f}, f(x)={optimizer.max['target']:.3f}")

In [ ]:
# Visualize all evaluated points
fig, ax = plt.subplots(figsize=(10, 5))

# True function
ax.plot(x_range, y_true, 'b-', linewidth=2, alpha=0.5, label='True function')

# Evaluated points
xs = [res['params']['x'] for res in optimizer.res]
ys = [res['target'] for res in optimizer.res]

# Color by order: early = light, late = dark
colors = plt.cm.Reds(np.linspace(0.3, 1.0, len(xs)))
for i, (x, y, c) in enumerate(zip(xs, ys, colors)):
    label = f'Point {i+1}' if i < 3 else (f'Point {i+1}' if i == 3 else None)
    marker = 's' if i < 3 else 'o'  # squares for random, circles for BO
    ax.scatter(x, y, c=[c], s=100, edgecolors='k', marker=marker, zorder=5)
    ax.annotate(f'{i+1}', (x, y), textcoords='offset points',
                xytext=(5, 10), fontsize=9, fontweight='bold')

# Mark the best
best_x = optimizer.max['params']['x']
best_y = optimizer.max['target']
ax.axvline(best_x, color='green', linestyle='--', alpha=0.5)
ax.scatter([best_x], [best_y], c='green', s=200, marker='*', zorder=6,
           edgecolors='k', label=f'Best: x={best_x:.2f}, f(x)={best_y:.2f}')

ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('f(x)', fontsize=13)
ax.set_title('Bayesian Optimization: 3 random + 7 guided evaluations', fontsize=14)
ax.legend(loc='lower left', fontsize=10)
plt.tight_layout()
plt.show()

print("Squares = random initialization, circles = Bayesian-guided")
print("Notice how later points cluster around the optimum.")

### 2b. Optuna (TPE-based Bayesian Optimization)

Optuna uses Tree Parzen Estimators (TPE) instead of Gaussian Processes. TPE scales better to many parameters and handles categorical variables.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Give Optuna MORE budget than grid/random to show its advantage
# In practice, Optuna shines when you have more trials to learn from
OPTUNA_BUDGET = 100  # 100 trials vs 36 for grid

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 30),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'max_features': trial.suggest_float('max_features', 0.1, 1.0),
    }
    model = RandomForestClassifier(**params, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=OPTUNA_BUDGET)

print(f"Optuna: {OPTUNA_BUDGET} trials")
print(f"Best CV score : {study.best_value:.4f}")
print(f"Best params   : {study.best_params}")

In [ ]:
# Optuna: show how score improves over trials
trial_values = [t.value for t in study.trials]
best_so_far = np.maximum.accumulate(trial_values)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(range(len(trial_values)), trial_values, alpha=0.3, s=20, label='Individual trials')
ax.plot(best_so_far, 'r-', linewidth=2, label='Best so far')
ax.axhline(grid_cv.best_score_, color='blue', linestyle='--', alpha=0.7,
           label=f'Grid best ({grid_cv.best_score_:.4f})')
ax.axhline(random_cv.best_score_, color='green', linestyle='--', alpha=0.7,
           label=f'Random best ({random_cv.best_score_:.4f})')
ax.set_xlabel('Trial number')
ax.set_ylabel('CV Accuracy')
ax.set_title('Optuna Optimization History')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Optuna built-in: parameter importances
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
# --- Compare all methods ---
methods = ['Grid Search\n(36 evals)', 'Random Search\n(36 evals)',
           f'Optuna (TPE)\n({OPTUNA_BUDGET} evals)']
scores = [grid_cv.best_score_, random_cv.best_score_, study.best_value]

print("=" * 55)
print(f"{'Method':<25} {'Budget':>8} {'Best CV Acc':>12}")
print("=" * 55)
for m, b, s in zip(['Grid', 'Random', 'Optuna (TPE)'],
                    [BUDGET, BUDGET, OPTUNA_BUDGET], scores):
    print(f"{m:<25} {b:>8} {s:>12.4f}")
print("=" * 55)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4C72B0', '#55A868', '#C44E52']
bars = ax.bar(methods, scores, color=colors, edgecolor='black', linewidth=0.8)
ax.set_ylabel('Best CV Accuracy')
ax.set_title('Tuning Methods Comparison')
ax.set_ylim(min(scores) - 0.01, max(scores) + 0.005)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 3: Nested Cross-Validation

`GridSearchCV.best_score_` is **optimistic** because it reports the best score found after searching -- this is selection bias. Nested CV gives an unbiased estimate.

In [ ]:
# --- The optimism gap ---

# Inner CV: tunes hyperparameters
inner_cv = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={'max_depth': [5, 10, 15, None], 'n_estimators': [50, 100, 200]},
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
)

# Outer CV: evaluates the tuned pipeline
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
outer_scores = cross_val_score(inner_cv, X, y, cv=outer_cv, scoring='accuracy')

# Also get the (optimistic) best_score_
inner_cv.fit(X, y)
optimistic_score = inner_cv.best_score_
nested_score = outer_scores.mean()
gap = optimistic_score - nested_score

print("=" * 55)
print(f"  GridSearchCV.best_score_ (OPTIMISTIC) : {optimistic_score:.4f}")
print(f"  Nested CV outer scores                : {outer_scores}")
print(f"  Nested CV mean (UNBIASED)             : {nested_score:.4f} +/- {outer_scores.std():.4f}")
print(f"  Optimism gap                          : {gap:.4f}")
print("=" * 55)

In [ ]:
# --- Visualize the gap ---
fig, ax = plt.subplots(figsize=(8, 4))

ax.barh(['Nested CV\n(unbiased)', 'GridSearchCV.best_score_\n(optimistic)'],
        [nested_score, optimistic_score],
        color=['#55A868', '#C44E52'], edgecolor='black', linewidth=0.8, height=0.5)

ax.errorbar(nested_score, 0, xerr=outer_scores.std(), fmt='none',
            ecolor='black', capsize=5, linewidth=2)

ax.set_xlabel('Accuracy')
ax.set_title(f'The Optimism Gap: {gap:.4f}')
ax.set_xlim(min(nested_score - outer_scores.std(), optimistic_score) - 0.02,
            max(nested_score, optimistic_score) + 0.02)

for i, v in enumerate([nested_score, optimistic_score]):
    ax.text(v + 0.003, i, f'{v:.4f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Part 4: AutoML with FLAML

FLAML (Microsoft) is a lightweight, fast AutoML library. Unlike AutoGluon, it has minimal dependencies and installs cleanly.

```
pip install flaml
```

In [ ]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train, y_train,
    task="classification",
    time_budget=60,         # 60 seconds
    metric="accuracy",
    verbose=0,
)

print(f"Best estimator : {automl.best_estimator}")
print(f"Best config    : {automl.best_config}")
print(f"Best CV score  : {1 - automl.best_loss:.4f}")
print(f"Test accuracy  : {automl.score(X_test, y_test):.4f}")

In [ ]:
# --- Complete workflow: Dummy -> LR -> Tuned RF -> AutoML ---
results = {}

# 1. Dummy baseline
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
results['Dummy (baseline)'] = dummy.score(X_test, y_test)

# 2. Logistic Regression (simple model)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
results['Logistic Regression'] = lr.score(X_test, y_test)

# 3. Tuned Random Forest
best_rf = grid_cv.best_estimator_
results['Tuned RF (GridSearch)'] = best_rf.score(X_test, y_test)

# 4. FLAML AutoML
results['FLAML AutoML'] = automl.score(X_test, y_test)

print(f"{'Model':<25} {'Test Accuracy':>15}")
print("=" * 42)
for name, acc in results.items():
    print(f"{name:<25} {acc:>15.4f}")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
names = list(results.keys())
accs = list(results.values())
colors = ['#cccccc', '#4C72B0', '#55A868', '#C44E52']
bars = ax.barh(names, accs, color=colors, edgecolor='black', linewidth=0.8)
ax.set_xlabel('Test Accuracy')
ax.set_title('Complete Workflow: Baselines through AutoML')
ax.set_xlim(0.4, 1.0)
for bar, acc in zip(bars, accs):
    ax.text(acc + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{acc:.4f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Part 5: PyTorch Reproducibility

In [ ]:
import torch
import torch.nn as nn
import random
import os

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

In [ ]:
def set_seed_full(seed=42):
    """Complete seed function for full PyTorch reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.use_deterministic_algorithms(True)

In [ ]:
# --- Without seeds: different output every run ---
print("=== WITHOUT seeds (outputs differ each run) ===")
outputs_no_seed = []
for i in range(3):
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y_pred = model(x)
    val = y_pred[0].item()
    outputs_no_seed.append(val)
    print(f"  Run {i+1}: first output = {val:.6f}")

print(f"  All same? {len(set([f'{v:.6f}' for v in outputs_no_seed])) == 1}")

In [ ]:
# --- With set_seed_full: identical output every run ---
print("=== WITH set_seed_full (outputs are identical) ===")
outputs_seeded = []
for i in range(3):
    set_seed_full(42)
    model = nn.Linear(10, 1)
    x = torch.randn(5, 10)
    y_pred = model(x)
    val = y_pred[0].item()
    outputs_seeded.append(val)
    print(f"  Run {i+1}: first output = {val:.6f}")

print(f"  All same? {len(set([f'{v:.6f}' for v in outputs_seeded])) == 1}")

In [ ]:
# --- Multi-seed reporting ---
print("=== Multi-Seed Reporting ===")
seed_results = []
seeds = [42, 123, 456, 789, 1024]

for seed in seeds:
    np.random.seed(seed)
    X_s, y_s = make_classification(
        n_samples=500, n_features=10, n_informative=5,
        n_redundant=2, random_state=seed
    )
    Xtr, Xte, ytr, yte = train_test_split(X_s, y_s, test_size=0.2, random_state=seed)
    m = RandomForestClassifier(n_estimators=100, random_state=seed)
    m.fit(Xtr, ytr)
    acc = m.score(Xte, yte)
    seed_results.append(acc)
    print(f"  Seed {seed:>4d}: accuracy = {acc:.4f}")

mean_acc = np.mean(seed_results)
std_acc = np.std(seed_results)
print(f"\nResult: {mean_acc:.4f} +/- {std_acc:.4f}")
print("This is more honest than reporting a single lucky seed.")

fig, ax = plt.subplots(figsize=(8, 3))
ax.barh([f'Seed {s}' for s in seeds], seed_results, color='#4C72B0',
        edgecolor='black', linewidth=0.5)
ax.axvline(mean_acc, color='red', linestyle='--', linewidth=2, label=f'Mean = {mean_acc:.4f}')
ax.set_xlabel('Test Accuracy')
ax.set_title('Multi-Seed Reporting')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 6: Experiment Tracking with Trackio

Trackio is a local-first, free experiment tracking library from Hugging Face. It has a W&B-compatible API.

```
pip install trackio
```

Launch the dashboard with:
```
trackio
```

In [ ]:
import trackio

# --- Run 1: Logistic Regression baseline ---
trackio.init(project="cs203-week08", run_name="logistic-baseline", config={
    "model": "LogisticRegression",
    "C": 1.0,
    "max_iter": 1000,
    "seed": 42,
})

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
train_acc = lr_model.score(X_train, y_train)
test_acc = lr_model.score(X_test, y_test)

trackio.log({"train_accuracy": train_acc, "test_accuracy": test_acc})
print(f"LR — Train: {train_acc:.4f}, Test: {test_acc:.4f}")
trackio.finish()

In [ ]:
# --- Run 2: Random Forest with tuning ---
trackio.init(project="cs203-week08", run_name="rf-tuned", config={
    "model": "RandomForest",
    "n_estimators": grid_cv.best_params_['n_estimators'],
    "max_depth": grid_cv.best_params_['max_depth'],
    "seed": 42,
})

rf_model = grid_cv.best_estimator_
train_acc_rf = rf_model.score(X_train, y_train)
test_acc_rf = rf_model.score(X_test, y_test)

trackio.log({"train_accuracy": train_acc_rf, "test_accuracy": test_acc_rf,
             "train_test_gap": train_acc_rf - test_acc_rf})
print(f"RF — Train: {train_acc_rf:.4f}, Test: {test_acc_rf:.4f}")
trackio.finish()

In [ ]:
# --- Run 3: Log metrics over "epochs" (simulated training loop) ---
trackio.init(project="cs203-week08", run_name="rf-incremental", config={
    "model": "RandomForest",
    "max_estimators": 200,
    "seed": 42,
})

# Simulate training with increasing number of trees
for n_trees in range(10, 210, 10):
    rf = RandomForestClassifier(n_estimators=n_trees, random_state=42)
    rf.fit(X_train, y_train)
    trackio.log({
        "n_estimators": n_trees,
        "train_accuracy": rf.score(X_train, y_train),
        "test_accuracy": rf.score(X_test, y_test),
    })

trackio.finish()
print("Logged 20 steps. Open 'trackio' dashboard to see the learning curve.")

In [ ]:
print("=" * 60)
print("To view your experiments, run in a terminal:")
print("  trackio")
print("")
print("This opens a local Gradio dashboard where you can:")
print("  - Compare runs side-by-side")
print("  - View metric curves (train_accuracy, test_accuracy)")
print("  - Filter by config values")
print("  - Export results")
print("")
print("Data is stored locally in ~/.cache/huggingface/trackio/")
print("No cloud account needed!")
print("=" * 60)

---
## Summary

| Topic | Key takeaway |
|-------|-------------|
| Grid vs Random | Random covers the space better with same budget |
| Bayesian Opt (GP) | Models the function, uses uncertainty to pick next point |
| Optuna (TPE) | Scales to many params, handles categorical, prunes bad trials |
| Nested CV | `best_score_` is optimistic; nested CV is unbiased |
| FLAML | Lightweight AutoML with cost-frugal search |
| Reproducibility | `set_seed_full` + deterministic algorithms |
| Multi-seed | Report mean +/- std across seeds |
| Trackio | Local-first, free experiment tracking with Gradio dashboard |